[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/Lecture1-NN-JAX.ipynb)

## Setup

Run the setup cell below once at the start of the session. A CPU runtime is sufficient. The data are generated in the notebook; no downloads or Google Drive mount are needed.

Run the cells in order and complete the marked exercises. The exercise placeholders deliberately raise `NotImplementedError` until you fill them in.

In [ ]:
# Install the required packages without upgrading packages already available.
#%pip -q install jax numpy matplotlib

import jax
print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())

# Neural networks with Jax!

![img](https://jax.readthedocs.io/en/latest/_static/jax_logo_250px.png)

Jax is an efficient computation package (think `numpy` with coprocessors) that includes automatic differentiation of functions. Jax has become quite popular in recent years for many different fields, including machine learning.

Let's start by exploring the parts of the package that are useful for us.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

Let's build a simple function for

$$f(x) = \sin{3x}$$

This is a function that we (hopefully) know the derivative for, so let's also code a function that returns the analytic derivative for comparison.

In [ ]:
# standard way of defining a python function
def f(x):
    return jnp.sin(3*x) # using jax numpy operations

# define a function for the analytic derivative (not using automatic differentiation)
def fprime(x):
    return 3*jnp.cos(3*x) # still can use jax numpy for faster operations

Now, let's use the automatic differentiation engine within Jax and compare it to a known derivative using `jax.grad()`

https://jax.readthedocs.io/en/latest/notebooks/quickstart.html#taking-derivatives-with-grad#

In [ ]:
# Transform f(x) into a function that computes its derivative.
diff_f = jax.grad(f)

print("autodiff derivative:", diff_f(5.))
print("analytic derivative:", fprime(5.))

You should observe that the autodiff result is *exact* to numerical roundoff. That is because this is an analytically exact method, not a numeric method. Try modifying the above cells with a different function.

Notice that `jax.grad` transforms a function. Therefore, `diff_f(x)` is a transformed function that computes the gradient (e.g. derivative), of `f(x)`.

---

## Now, let's create some data for a regression task.

I will create a synthetic dataset with three features that map to our ground truth output $y$ via the function
$$y = 0.5x_1 + 0.2x_2 + 2x_3 + \sigma,$$ where $\sigma$ is independent Gaussian noise with mean 0 and standard deviation 0.1 for each example.

In [ ]:
# Define reproducible synthetic data.
rng = np.random.default_rng(42)
N = 1000
n_feat = 3
x = rng.uniform(size=(N, n_feat))

def true(x):
    noise = rng.normal(scale=0.1, size=len(x))
    return 0.5*x[:, 0] + 0.2*x[:, 1] + 2*x[:, 2] + noise

y = true(x)

In [ ]:
#checking that the data looks as expected
#plt.plot(x[:,0], y,'.')
#plt.show()

## Let's first build a linear regression model, and find the optimal model parameters via gradient descent.

We know that the data follows a linear trend, **and** we know the coefficients. This is a great testing ground for this package. 

We are going to build the same algorithm as we did for linear regression via gradient descent in the lecture, but we will use automatic differentiation for computing gradients.

To maximize efficiency, I am going to build a model that will simultaneously predict on the entire training dataset. To do this, I need some linear algebra.


$ \mathbf{x} $ with 4 examples and 3 features, is represented as:
$$
\mathbf{x} = \begin{bmatrix}
x_{11} & x_{12} & x_{13} \\
x_{21} & x_{22} & x_{23} \\
x_{31} & x_{32} & x_{33} \\
x_{41} & x_{42} & x_{43}
\end{bmatrix}
$$
And our model weights vector
$$
\mathbf{w} = \begin{bmatrix}
w_1 \\
w_2 \\
w_3
\end{bmatrix}
$$



The model $$ \mathbf{y_{pred}} = \mathbf{x} \cdot \mathbf{w} $$ will provide a $4 \times 1$ array of predictions, one for each example.

$$
\mathbf{y_{pred}} = \begin{bmatrix}
w_1x_{11} + w_2x_{12} + w_3x_{13} \\
w_1x_{21} + w_2x_{22} + w_3x_{23} \\
w_1x_{31} + w_2x_{32} + w_3x_{33} \\
w_1x_{41} + w_2x_{42} + w_3x_{43}
\end{bmatrix}
$$

(*Note: I left out the bias term, which is not good practice but ok for our simple example because the ground truth model has no bias.*)

In [ ]:
# Define the linear regression model described above.
def model(w, x):
    return jnp.dot(x, w)

# Exercise: return the mean squared error. jnp.mean() can help.
def loss(w, x, y_tr):
    raise NotImplementedError("Complete the mean squared error loss.")

# Exercise: define loss_gradient using jax.grad.

# Optimize the parameters and track training and validation losses.
def gradient_descent(w, xtrain, ytrain, xval, yval,
                     learning_rate=0.1, num_epochs=100):
    loss_f = np.zeros(num_epochs)
    valloss_f = np.zeros(num_epochs)
    for i in range(num_epochs):
        # Exercise: compute the gradient and update w.
        raise NotImplementedError("Complete the gradient descent step.")

        loss_f[i] = loss(w, xtrain, ytrain)
        valloss_f[i] = loss(w, xval, yval)

    return w, loss_f, valloss_f

In [ ]:
# Initialize parameters randomly.
key = jax.random.PRNGKey(5)
w = jax.random.normal(key, (n_feat,))
split = int(0.8 * N)

# Training and validation data.
train_x = x[:split]
train_y = y[:split]
val_x = x[split:]
val_y = y[split:]

# Exercise: generate independent test_x and test_y using rng and true().

# Perform gradient descent.
w, lossf, val_lossf = gradient_descent(
    w, train_x, train_y, val_x, val_y, learning_rate=0.1, num_epochs=500
)
plt.plot(lossf, label="train loss")
plt.plot(val_lossf, label="val loss")
plt.xlabel("epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.show()
print("Learned parameters:", w)

# Exercise: predict on test_x and print the test mean squared error.

### Awesome! You should have gotten decent results, and the code should have been simpler and faster to execute than our first go at linear regression.
 ---

# Neural Networks

Let's now apply what we have learned to build a simple regression neural network with 1 hidden layer of 4 nodes.

The trick here is to create a 2D **matrix** of weights, where number of rows is the number weights in the model and the number of columns is the number of nodes in the layer. This allows us to operate on the entire *layer* at once.

In [ ]:
# Sigmoid activation for the hidden layer.
def sigmoid_layer(sig_w, x):
    # Exercise: apply jax.nn.sigmoid to jnp.dot(x, sig_w).
    raise NotImplementedError("Complete the sigmoid layer.")

# Linear output layer for regression.
def linear(lin_w, inputs):
    # Exercise: use jnp.dot.
    raise NotImplementedError("Complete the linear output layer.")

def nn_model(ws, x):
    w_sig = ws[0]
    w_lin = ws[1]
    # Exercise: pass x through the hidden layer and then the output layer.
    raise NotImplementedError("Complete the neural network model.")

# Exercise: return the mean squared error of the neural network predictions.
def nn_loss(w, x, y):
    raise NotImplementedError("Complete the neural network loss.")

# Exercise: define nn_loss_gradient using jax.grad.

def nn_gradient_descent(w, xtrain, ytrain, xval, yval,
                        learning_rate=0.1, num_epochs=1000):
    w1, w2 = w
    loss_f = np.zeros(num_epochs)
    valloss_f = np.zeros(num_epochs)
    for i in range(num_epochs):
        # Exercise: compute both weight gradients and update w1 and w2.
        raise NotImplementedError("Complete both neural network weight updates.")
        w = [w1, w2]

        loss_f[i] = nn_loss(w, xtrain, ytrain)
        valloss_f[i] = nn_loss(w, xval, yval)

    return w, loss_f, valloss_f

### Follow your linear regression example to train your (maybe) first neural network!

We will use the same data as our linear model above. I have initialized the neural network weights for you, but the rest is up to you!

Make sure to plot your loss curve and make predictions on your test set.

Play around! Change hyperparameters, change the data, have fun! 🎉

In [ ]:
# Initialize the two layers with independent random keys.
key = jax.random.PRNGKey(22)
key1, key2 = jax.random.split(key)
num_features = n_feat
num_hidden = 4

# Layer 1: num_features inputs and num_hidden nodes.
w1 = jax.random.normal(key1, (num_features, num_hidden))

# Layer 2: num_hidden inputs and one output.
w2 = jax.random.normal(key2, (num_hidden,))

In [ ]:
# Exercise: train the network using nn_gradient_descent([w1, w2], ...).
# Plot the training and validation loss curves.
# Predict on the same independent test set and print the test mean squared error.
# Plot predicted y against test_y.